In [4]:
import io
import json
import time
import statistics

import torch
import torch.nn as nn
from torch.ao.nn.quantized.dynamic import Linear as DynamicQuantizedLinear
from transformers import pipeline

from dataset import CANDIDATE_LABELS, TEXTS, TRUE_LABELS


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0916 17:15:54.048000 2788 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0916 17:15:54.149000 2788 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [3]:

def benchmark_model(model_name, clf, texts, true_labels):
    predictions = []
    latencies = []

    # Warm-up
    clf(texts[0], CANDIDATE_LABELS)

    # Benchmark
    for text in texts:
        start = time.perf_counter()

        result = clf(text, CANDIDATE_LABELS)

        end = time.perf_counter()

        latencies.append(end - start)
        predictions.append(result["labels"][0])

    # Metrics
    correct = sum(
        p == t for p, t in zip(predictions, true_labels)
    )

    accuracy = correct / len(true_labels)

    return {
        "model": model_name,
        "accuracy": accuracy,
        "mean_latency_ms": statistics.mean(latencies) * 1000,
        "median_latency_ms": statistics.median(latencies) * 1000,
        "p95_latency_ms": statistics.quantiles(
            latencies, n=20
        )[18] * 1000,
        "throughput": len(texts) / sum(latencies),
        "predictions": predictions,
    }

In [10]:
def get_serialized_size_mb(model):
    buffer = io.BytesIO()
    torch.save(model.state_dict(), buffer)
    return buffer.getbuffer().nbytes / (1024 ** 2)

FP32 MODEL

In [5]:
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=-1,
)
print("Loaded.")

Device set to use cpu


Loaded.


In [6]:
fp32_classifier = classifier

In [ ]:
fp32_results = benchmark_model(
    "FP32",
    fp32_classifier,
    TEXTS,
    TRUE_LABELS
)



{'model': 'FP32', 'accuracy': 0.85, 'mean_latency_ms': 854.9424075002207, 'median_latency_ms': 805.8542999988276, 'p95_latency_ms': 1260.2733799994894, 'throughput': 1.1696694317970673, 'predictions': ['world news', 'world news', 'world news', 'world news', 'world news', 'world news', 'world news', 'business', 'world news', 'world news', 'sports', 'sports', 'sports', 'sports', 'sports', 'sports', 'sports', 'sports', 'sports', 'sports', 'business', 'business', 'business', 'world news', 'business', 'science and technology', 'business', 'world news', 'business', 'world news', 'science and technology', 'science and technology', 'science and technology', 'science and technology', 'science and technology', 'science and technology', 'world news', 'science and technology', 'science and technology', 'science and technology']}


In [13]:
print(fp32_results)
fp32_results["model_size_mb"] = get_serialized_size_mb(fp32_classifier.model)
print(f"Accuracy:            {fp32_results['accuracy']*100:.1f}%")
print(f"Mean latency:        {fp32_results['mean_latency_ms']*1000:.1f} ms")
print(f"Median latency:      {fp32_results['median_latency_ms']*1000:.1f} ms")
print(f"P95 latency:         {fp32_results['p95_latency_ms']*1000:.1f} ms")
print(f"Throughput:          {fp32_results['throughput']:.2f} examples/sec")
print(f"Model size (INT8):   {fp32_results['model_size_mb']:.2f} MB")

{'model': 'FP32', 'accuracy': 0.85, 'mean_latency_ms': 854.9424075002207, 'median_latency_ms': 805.8542999988276, 'p95_latency_ms': 1260.2733799994894, 'throughput': 1.1696694317970673, 'predictions': ['world news', 'world news', 'world news', 'world news', 'world news', 'world news', 'world news', 'business', 'world news', 'world news', 'sports', 'sports', 'sports', 'sports', 'sports', 'sports', 'sports', 'sports', 'sports', 'sports', 'business', 'business', 'business', 'world news', 'business', 'science and technology', 'business', 'world news', 'business', 'world news', 'science and technology', 'science and technology', 'science and technology', 'science and technology', 'science and technology', 'science and technology', 'world news', 'science and technology', 'science and technology', 'science and technology'], 'model_size_mb': 1554.0723810195923}
Accuracy:            85.0%
Mean latency:        854942.4 ms
Median latency:      805854.3 ms
P95 latency:         1260273.4 ms
Through

DYNAMIC INT8 MODEL

In [8]:
# applying the dynamic quntization
quantized_model = torch.quantization.quantize_dynamic(
    classifier.model,
    {nn.Linear},
    dtype=torch.qint8,
)

print("Quantization applied.")


C:\Users\Admin\AppData\Local\Temp\ipykernel_2788\680331107.py:2: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


Quantization applied.


In [9]:
# DYNAMIC INT8 MODEL
dynamic_int8_classifier = pipeline(
    "zero-shot-classification",
    model=quantized_model,
    tokenizer=classifier.tokenizer,
    device=-1
)

dynamic_int8_results = benchmark_model(
    "Dynamic INT8",
    dynamic_int8_classifier,
    TEXTS,
    TRUE_LABELS
)



Device set to use cpu


In [14]:
print(dynamic_int8_results)
dynamic_int8_results["model_size_mb"] = get_serialized_size_mb(dynamic_int8_classifier.model)
print(f"Accuracy:            {dynamic_int8_results['accuracy']*100:.1f}%")
print(f"Mean latency:        {dynamic_int8_results['mean_latency_ms']*1000:.1f} ms")
print(f"Median latency:      {dynamic_int8_results['median_latency_ms']*1000:.1f} ms")
print(f"P95 latency:         {dynamic_int8_results['p95_latency_ms']*1000:.1f} ms")
print(f"Throughput:          {dynamic_int8_results['throughput']:.2f} examples/sec")
print(f"Model size (INT8):   {dynamic_int8_results['model_size_mb']:.2f} MB")

{'model': 'Dynamic INT8', 'accuracy': 0.35, 'mean_latency_ms': 2078.436805000001, 'median_latency_ms': 2168.2661499999085, 'p95_latency_ms': 2607.938235000802, 'throughput': 0.4811308179273699, 'predictions': ['science and technology', 'science and technology', 'business', 'business', 'business', 'sports', 'science and technology', 'world news', 'sports', 'business', 'business', 'business', 'science and technology', 'business', 'business', 'science and technology', 'business', 'sports', 'sports', 'world news', 'sports', 'world news', 'business', 'world news', 'business', 'world news', 'business', 'science and technology', 'science and technology', 'business', 'world news', 'science and technology', 'science and technology', 'business', 'science and technology', 'science and technology', 'science and technology', 'science and technology', 'world news', 'science and technology']}
Accuracy:            35.0%
Mean latency:        2078436.8 ms
Median latency:      2168266.1 ms
P95 latency:  

WEIGHT-ONLY INT8 MODEL

In [15]:
import copy
import torch

from torchao.quantization import quantize_, Int8WeightOnlyConfig

# Start from the original FP32 model
weight_only_model = copy.deepcopy(classifier.model)
weight_only_model.eval()

# INT8 weight-only quantization
quantize_(
    weight_only_model,
    Int8WeightOnlyConfig()
)

weight_only_model.eval()

print("Weight-only INT8 model created!")

Weight-only INT8 model created!


In [16]:
weight_only_classifier = pipeline(
    "zero-shot-classification",
    model=weight_only_model,
    tokenizer=classifier.tokenizer,
    device=-1
)



Device set to use cpu


In [17]:
weight_only_results = benchmark_model(
    "Weight-only INT8",
    weight_only_classifier,
    TEXTS,
    TRUE_LABELS
)



In [18]:
print(weight_only_results)
weight_only_results["model_size_mb"] = get_serialized_size_mb(weight_only_classifier.model)
print(f"Accuracy:            {weight_only_results['accuracy']*100:.1f}%")
print(f"Mean latency:        {weight_only_results['mean_latency_ms']*1000:.1f} ms")
print(f"Median latency:      {weight_only_results['median_latency_ms']*1000:.1f} ms")
print(f"P95 latency:         {weight_only_results['p95_latency_ms']*1000:.1f} ms")
print(f"Throughput:          {weight_only_results['throughput']:.2f} examples/sec")
print(f"Model size (INT8):   {weight_only_results['model_size_mb']:.2f} MB")

{'model': 'Weight-only INT8', 'accuracy': 0.875, 'mean_latency_ms': 1596.4434650002659, 'median_latency_ms': 1606.9030500002555, 'p95_latency_ms': 1894.96174500091, 'throughput': 0.6263923664843549, 'predictions': ['world news', 'world news', 'world news', 'world news', 'world news', 'world news', 'world news', 'business', 'world news', 'world news', 'sports', 'sports', 'sports', 'sports', 'sports', 'sports', 'sports', 'sports', 'sports', 'sports', 'business', 'business', 'business', 'world news', 'business', 'science and technology', 'business', 'world news', 'business', 'world news', 'science and technology', 'science and technology', 'science and technology', 'science and technology', 'science and technology', 'science and technology', 'science and technology', 'science and technology', 'science and technology', 'science and technology']}
Accuracy:            87.5%
Mean latency:        1596443.5 ms
Median latency:      1606903.1 ms
P95 latency:         1894961.7 ms
Throughput:      

In [19]:
results = [
    fp32_results,
    dynamic_int8_results,
    weight_only_results
]

print("\n===== MODEL COMPARISON =====\n")

for r in results:
    print(f"{r['model']}")
    print(f"  Accuracy:       {r['accuracy'] * 100:.1f}%")
    print(f"  Mean latency:   {r['mean_latency_ms']:.1f} ms")
    print(f"  Median latency: {r['median_latency_ms']:.1f} ms")
    print(f"  P95 latency:    {r['p95_latency_ms']:.1f} ms")
    print(f"  Throughput:     {r['throughput']:.2f} examples/sec")
    print()


===== MODEL COMPARISON =====

FP32
  Accuracy:       85.0%
  Mean latency:   854.9 ms
  Median latency: 805.9 ms
  P95 latency:    1260.3 ms
  Throughput:     1.17 examples/sec

Dynamic INT8
  Accuracy:       35.0%
  Mean latency:   2078.4 ms
  Median latency: 2168.3 ms
  P95 latency:    2607.9 ms
  Throughput:     0.48 examples/sec

Weight-only INT8
  Accuracy:       87.5%
  Mean latency:   1596.4 ms
  Median latency: 1606.9 ms
  P95 latency:    1895.0 ms
  Throughput:     0.63 examples/sec



In [20]:
import time
import numpy as np


def benchmark_batched(clf, texts, true_labels, batch_size=8):

    predictions = []
    latencies = []

    # Warmup
    clf(texts[:batch_size], CANDIDATE_LABELS)

    for i in range(0, len(texts), batch_size):

        batch_texts = texts[i:i + batch_size]

        start = time.perf_counter()

        results = clf(
            batch_texts,
            CANDIDATE_LABELS,
            batch_size=batch_size
        )

        end = time.perf_counter()

        batch_latency = end - start

        # Store total batch latency
        latencies.append(batch_latency)

        # Get prediction for each text
        predictions.extend(
            [result["labels"][0] for result in results]
        )

    accuracy = np.mean(
        np.array(predictions) == np.array(true_labels)
    )

    total_time = sum(latencies)

    return {
        "accuracy": accuracy,
        "total_time_sec": total_time,
        "mean_batch_latency_sec": np.mean(latencies),
        "throughput_examples_per_sec": len(texts) / total_time,
        "predictions": predictions
    }

In [21]:
batched_results = benchmark_batched(
    weight_only_classifier,
    TEXTS,
    TRUE_LABELS,
    batch_size=8
)

print("===== BATCHED WEIGHT-ONLY INT8 =====")

print(
    f"Accuracy:    "
    f"{batched_results['accuracy'] * 100:.1f}%"
)

print(
    f"Batch latency: "
    f"{batched_results['mean_batch_latency_sec'] * 1000:.1f} ms"
)

print(
    f"Throughput:  "
    f"{batched_results['throughput_examples_per_sec']:.2f} examples/sec"
)

===== BATCHED WEIGHT-ONLY INT8 =====
Accuracy:    87.5%
Batch latency: 4213.7 ms
Throughput:  1.90 examples/sec


In [22]:
for batch_size in [4, 8, 16]:

    result = benchmark_batched(
        weight_only_classifier,
        TEXTS,
        TRUE_LABELS,
        batch_size=batch_size
    )

    print(
        f"Batch {batch_size}: "
        f"{result['throughput_examples_per_sec']:.2f} examples/sec"
    )

Batch 4: 1.46 examples/sec
Batch 8: 1.82 examples/sec
Batch 16: 2.09 examples/sec
